# Multi-Analog Zone Blend Screen

Second alignment-track experiment. The first screen found useful but sub-gate signal from a single nearest analog. This notebook tests whether distance-weighted multi-analog transfer and pseudo-zone-specific blending improve that signal.

## 1. Setup

In [1]:
"Configure imports, paths, and candidate constants."

from pathlib import Path
import json
import shutil
import time

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "context").exists():
    PROJECT_ROOT = next(parent for parent in PROJECT_ROOT.parents if (parent / "context").exists())

DATE_STAMP = "2026-06-23"
RUN_ID = f"{DATE_STAMP}__multi_analog_zone_blend"
ARTIFACT_TABLES = PROJECT_ROOT / "artifacts" / "tables"
ARTIFACT_REPORTS = PROJECT_ROOT / "artifacts" / "reports"
ARTIFACT_LOGS = PROJECT_ROOT / "artifacts" / "logs" / "runs" / RUN_ID
for path in [ARTIFACT_TABLES, ARTIFACT_REPORTS, ARTIFACT_LOGS]:
    path.mkdir(parents=True, exist_ok=True)

RESIDUAL_PATH = ARTIFACT_TABLES / "model__hsgp_residual_on_gam_huber_residual_sample__2026-06-22.csv"
DESCRIPTOR_PATH = ARTIFACT_TABLES / "model__typewell_alignment_feature_descriptors__2026-06-23.csv"
TARGET_CANDIDATE = "huber_plus_hsgp_dps_m8"
LOCAL_REFERENCE_RMSE = 14.3586
PUBLIC_REFERENCE_RMSE = 14.514
OFFSET_CLIP = 12.0

ANALOG_DESCRIPTOR_FEATURES = [
    "known_tvt_span",
    "known_md_span",
    "known_gr_mean_last300",
    "known_gr_std_last300",
    "known_gr_slope_last300",
    "typewell_corr_last300",
    "typewell_best_lag_corr",
    "typewell_best_lag_tvt",
    "xy_span_known",
    "z_drop_known",
]

CANDIDATES = []
for k in [1, 3, 5, 8, 12]:
    for shrink in [0.05, 0.10, 0.15, 0.20]:
        CANDIDATES.append({"name": f"weighted_k{k}_s{int(shrink * 1000):03d}", "kind": "fixed_shrink", "k": k, "shrink": shrink})
for k in [1, 3, 5, 8, 12]:
    for cap in [0.10, 0.15, 0.20, 0.25]:
        CANDIDATES.append({"name": f"zone_beta_k{k}_cap{int(cap * 1000):03d}", "kind": "zone_beta", "k": k, "cap": cap})

print(f"project_root={PROJECT_ROOT}")
print(f"run_id={RUN_ID}")
print(f"candidate_count={len(CANDIDATES)}")

project_root=C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling
run_id=2026-06-23__multi_analog_zone_blend
candidate_count=40


## 2. Load Residuals And Descriptors

In [2]:
"Load the residual sample and descriptor table from the first alignment screen."

residuals_all = pd.read_csv(RESIDUAL_PATH)
residuals = residuals_all.loc[residuals_all["candidate"].eq(TARGET_CANDIDATE)].copy()
for col in ["fold", "row_idx", "eval_progress", "distance_from_boundary_md", "abs_projected_delta_10", "y_true", "y_pred", "residual"]:
    residuals[col] = pd.to_numeric(residuals[col], errors="coerce")
residuals["fold"] = residuals["fold"].astype(int)
residuals["row_idx"] = residuals["row_idx"].astype(int)
residuals["baseline_pred"] = residuals["y_pred"]
residuals["baseline_residual"] = residuals["y_true"] - residuals["baseline_pred"]
residuals["pseudo_zone"] = pd.cut(residuals["eval_progress"], bins=[-0.001, 0.25, 0.50, 0.75, 1.001], labels=["z0", "z1", "z2", "z3"])

descriptors = pd.read_csv(DESCRIPTOR_PATH)
model_frame = residuals.merge(descriptors[["wellname", "last_known_tvt"] + ANALOG_DESCRIPTOR_FEATURES], on="wellname", how="left")
model_frame["baseline_delta_from_last_known"] = model_frame["baseline_pred"] - model_frame["last_known_tvt"]
model_frame["true_delta_from_last_known"] = model_frame["y_true"] - model_frame["last_known_tvt"]

print(f"rows={len(model_frame):,}")
print(f"wells={model_frame['wellname'].nunique():,}")
print(model_frame.groupby("pseudo_zone", observed=False).size().rename("rows_by_zone").to_string())

rows=5,400
wells=773
pseudo_zone
z0    1160
z1    1414
z2    1412
z3    1414


## 3. Fold-Safe Multi-Analog Curves

In [3]:
"Build distance-weighted analog delta predictions for every fold and k value."


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def median_impute_scale(train_desc, all_desc, feature_cols):
    med = train_desc[feature_cols].median(numeric_only=True)
    scale = train_desc[feature_cols].std(numeric_only=True).replace(0, np.nan).fillna(1.0)
    train_scaled = (train_desc[feature_cols].fillna(med) - med) / scale
    all_scaled = (all_desc[feature_cols].fillna(med) - med) / scale
    return train_scaled, all_scaled


def interpolate_delta(train_rows, analog_well, progress_values):
    curve = train_rows.loc[train_rows["wellname"].eq(analog_well), ["eval_progress", "true_delta_from_last_known"]].dropna().sort_values("eval_progress")
    if len(curve) < 2:
        return np.full(len(progress_values), np.nan)
    grouped = curve.groupby("eval_progress", as_index=False)["true_delta_from_last_known"].mean()
    return np.interp(np.asarray(progress_values, dtype=float), grouped["eval_progress"].to_numpy(), grouped["true_delta_from_last_known"].to_numpy(), left=grouped["true_delta_from_last_known"].iloc[0], right=grouped["true_delta_from_last_known"].iloc[-1])


def weighted_analog_delta(train_rows, analog_names, analog_distances, progress_values, k):
    finite = np.isfinite(analog_distances)
    names = np.asarray(analog_names)[finite]
    distances = np.asarray(analog_distances, dtype=float)[finite]
    if len(names) == 0:
        return np.full(len(progress_values), np.nan), np.nan
    order = np.argsort(distances)[:k]
    picked_names = names[order]
    picked_distances = distances[order]
    weights = 1.0 / np.maximum(picked_distances, 1e-6)
    weights = weights / weights.sum()
    curves = []
    for analog_well in picked_names:
        curves.append(interpolate_delta(train_rows, analog_well, progress_values))
    curve_matrix = np.vstack(curves)
    return np.nansum(curve_matrix * weights[:, None], axis=0), float(np.mean(picked_distances))

k_values = sorted({candidate["k"] for candidate in CANDIDATES})
analog_chunks = []
for fold in sorted(model_frame["fold"].unique()):
    train_rows = model_frame.loc[~model_frame["fold"].eq(fold)].copy()
    valid_rows = model_frame.loc[model_frame["fold"].eq(fold)].copy()
    train_desc = descriptors.loc[descriptors["wellname"].isin(train_rows["wellname"].unique())].copy()
    valid_desc = descriptors.loc[descriptors["wellname"].isin(valid_rows["wellname"].unique())].copy()
    combined_desc = pd.concat([train_desc, valid_desc], ignore_index=True)
    train_scaled, all_scaled = median_impute_scale(train_desc, combined_desc, ANALOG_DESCRIPTOR_FEATURES)
    train_scaled.index = train_desc["wellname"].to_numpy()
    all_scaled.index = combined_desc["wellname"].to_numpy()
    train_names = train_scaled.index.to_numpy()
    train_matrix = train_scaled.to_numpy(dtype=float)
    for source in [train_rows, valid_rows]:
        source_kind = "train" if source is train_rows else "valid"
        for wellname, rows in source.groupby("wellname"):
            query = all_scaled.loc[wellname].to_numpy(dtype=float)
            distances = np.sqrt(np.mean((train_matrix - query) ** 2, axis=1))
            if source_kind == "train":
                distances = distances.copy()
                distances[train_names == wellname] = np.inf
            out = rows[["fold", "wellname", "row_idx", "eval_progress"]].copy()
            for k in k_values:
                analog_delta, mean_distance = weighted_analog_delta(train_rows, train_names, distances, rows["eval_progress"].to_numpy(), k)
                out[f"analog_delta_k{k}"] = analog_delta
                out[f"analog_distance_k{k}"] = mean_distance
            analog_chunks.append(out)

analog_frame = pd.concat(analog_chunks, ignore_index=True).drop_duplicates(["fold", "wellname", "row_idx"], keep="last")
model_frame = model_frame.merge(analog_frame, on=["fold", "wellname", "row_idx", "eval_progress"], how="left")
for k in k_values:
    model_frame[f"analog_gap_k{k}"] = model_frame[f"analog_delta_k{k}"] - model_frame["baseline_delta_from_last_known"]

print(model_frame[["wellname", "row_idx", "analog_gap_k1", "analog_gap_k3", "analog_gap_k5"]].head().to_string(index=False))

wellname  row_idx  analog_gap_k1  analog_gap_k3  analog_gap_k5
000d7d20     1442       5.418650       2.659964      -0.725129
000d7d20     1993      21.817007      10.933377       4.483033
000d7d20     2545      43.971705      22.884072      13.955216
000d7d20     3097      39.039547      20.877999      12.605248
000d7d20     3649      21.099572      16.721885       9.208891


## 4. Evaluate Fixed And Zone-Specific Blends

In [4]:
"Score fixed-shrink and pseudo-zone beta analog blends out of fold."

start_time = time.time()
prediction_rows = []
metric_rows = []
beta_rows = []
for candidate in CANDIDATES:
    k = candidate["k"]
    gap_col = f"analog_gap_k{k}"
    for fold in sorted(model_frame["fold"].unique()):
        train_rows = model_frame.loc[~model_frame["fold"].eq(fold)].copy()
        valid_rows = model_frame.loc[model_frame["fold"].eq(fold)].copy()
        if candidate["kind"] == "fixed_shrink":
            correction = candidate["shrink"] * valid_rows[gap_col].fillna(0.0).to_numpy()
        else:
            correction = np.zeros(len(valid_rows), dtype=float)
            for zone in ["z0", "z1", "z2", "z3"]:
                train_zone = train_rows.loc[train_rows["pseudo_zone"].astype(str).eq(zone)].copy()
                valid_mask = valid_rows["pseudo_zone"].astype(str).eq(zone).to_numpy()
                x = train_zone[gap_col].to_numpy(dtype=float)
                y = train_zone["baseline_residual"].to_numpy(dtype=float)
                mask = np.isfinite(x) & np.isfinite(y)
                if mask.sum() < 20 or np.sum(x[mask] ** 2) < 1e-9:
                    beta = 0.0
                else:
                    beta = float(np.sum(x[mask] * y[mask]) / np.sum(x[mask] ** 2))
                    beta = float(np.clip(beta, -candidate["cap"], candidate["cap"]))
                correction[valid_mask] = beta * valid_rows.loc[valid_mask, gap_col].fillna(0.0).to_numpy()
                beta_rows.append({"candidate": candidate["name"], "fold": int(fold), "zone": zone, "beta": beta})
        correction = np.clip(correction, -OFFSET_CLIP, OFFSET_CLIP)
        y_pred = valid_rows["baseline_pred"].to_numpy() + correction
        y_true = valid_rows["y_true"].to_numpy()
        metric_rows.append(
            {
                "candidate": candidate["name"],
                "kind": candidate["kind"],
                "fold": int(fold),
                "rmse": rmse(y_true, y_pred),
                "mae": float(mean_absolute_error(y_true, y_pred)),
                "n_rows": int(len(valid_rows)),
                "n_wells": int(valid_rows["wellname"].nunique()),
                "mean_correction": float(np.mean(correction)),
                "std_correction": float(np.std(correction)),
                "max_abs_correction": float(np.max(np.abs(correction))) if len(correction) else 0.0,
            }
        )
        out = valid_rows[["fold", "wellname", "row_idx", "pseudo_zone", "y_true", "baseline_pred", "baseline_residual", "eval_progress"]].copy()
        out["candidate"] = candidate["name"]
        out["analog_gap"] = valid_rows[gap_col].to_numpy()
        out["analog_correction"] = correction
        out["y_pred"] = y_pred
        out["residual_after_analog"] = out["y_true"] - out["y_pred"]
        prediction_rows.append(out)

fold_metrics = pd.DataFrame(metric_rows)
predictions = pd.concat(prediction_rows, ignore_index=True)
beta_summary = pd.DataFrame(beta_rows)
summary = (
    fold_metrics.groupby(["candidate", "kind"], as_index=False)
    .agg(
        rmse_mean=("rmse", "mean"),
        rmse_std=("rmse", "std"),
        mae_mean=("mae", "mean"),
        mean_correction=("mean_correction", "mean"),
        std_correction=("std_correction", "mean"),
        max_abs_correction=("max_abs_correction", "max"),
        n_rows=("n_rows", "sum"),
        n_wells=("n_wells", "sum"),
    )
)
baseline_rmse = rmse(model_frame["y_true"], model_frame["baseline_pred"])
summary["rmse_delta_vs_sample_baseline"] = summary["rmse_mean"] - baseline_rmse
summary = summary.sort_values(["rmse_mean", "candidate"]).reset_index(drop=True)
train_time_s = time.time() - start_time
print(summary.head(12).to_string(index=False))
print(f"sample_baseline_rmse={baseline_rmse:.4f}")

           candidate         kind  rmse_mean  rmse_std  mae_mean  mean_correction  std_correction  max_abs_correction  n_rows  n_wells  rmse_delta_vs_sample_baseline
   weighted_k12_s150 fixed_shrink  14.703434  0.271844 10.364775         0.041220        1.189620           12.000000    5400      773                      -0.040475
   weighted_k12_s100 fixed_shrink  14.704025  0.340351 10.371938         0.026664        0.800659            9.165688    5400      773                      -0.039884
    weighted_k1_s050 fixed_shrink  14.706869  0.406753 10.391269         0.016333        0.820072            4.233987    5400      773                      -0.037040
    weighted_k8_s100 fixed_shrink  14.708385  0.337773 10.374551         0.020191        0.857981            9.262463    5400      773                      -0.035524
    weighted_k8_s150 fixed_shrink  14.712415  0.268367 10.370543         0.031680        1.274701           12.000000    5400      773                      -0.031495
   w

## 5. Diagnostics And Export

In [5]:
"Export result tables, report, and run bundle."

best_candidate = summary.iloc[0]["candidate"]
best_row = summary.iloc[0]
best_predictions = predictions.loc[predictions["candidate"].eq(best_candidate)].copy()
well_before = model_frame.groupby(["fold", "wellname"], as_index=False).agg(mean_residual_before=("baseline_residual", "mean"))
well_after = best_predictions.groupby(["fold", "wellname"], as_index=False).agg(mean_residual_after=("residual_after_analog", "mean"), mean_analog_correction=("analog_correction", "mean"))
well_diagnostics = well_after.merge(well_before, on=["fold", "wellname"], how="left")
well_diagnostics["abs_bias_delta"] = well_diagnostics["mean_residual_after"].abs() - well_diagnostics["mean_residual_before"].abs()

findings = pd.DataFrame(
    [
        {"finding": "best_candidate", "value": f"{best_candidate} RMSE {best_row['rmse_mean']:.4f} vs sample baseline {baseline_rmse:.4f}", "modeling_implication": "Promote only if the 0.05 RMSE gate clears and fold diagnostics are stable."},
        {"finding": "sample_baseline_delta", "value": f"best candidate delta {best_row['rmse_delta_vs_sample_baseline']:.4f}", "modeling_implication": "A stronger direct analog gain supports a Kaggle-safe analog feature build."},
        {"finding": "well_bias_change", "value": f"median absolute well-bias delta {well_diagnostics['abs_bias_delta'].median():.4f}", "modeling_implication": "Negative is good; positive indicates analog blending is adding noise."},
    ]
)

summary_path = ARTIFACT_TABLES / f"model__multi_analog_zone_blend_summary__{DATE_STAMP}.csv"
fold_path = ARTIFACT_TABLES / f"model__multi_analog_zone_blend_fold_metrics__{DATE_STAMP}.csv"
pred_path = ARTIFACT_TABLES / f"model__multi_analog_zone_blend_residual_sample__{DATE_STAMP}.csv"
well_path = ARTIFACT_TABLES / f"model__multi_analog_zone_blend_well_diagnostics__{DATE_STAMP}.csv"
beta_path = ARTIFACT_TABLES / f"model__multi_analog_zone_blend_betas__{DATE_STAMP}.csv"
findings_path = ARTIFACT_TABLES / f"model__multi_analog_zone_blend_findings__{DATE_STAMP}.csv"
report_path = ARTIFACT_REPORTS / f"model__multi_analog_zone_blend__{DATE_STAMP}.md"
comparison_path = ARTIFACT_TABLES / "model_comparison.csv"

summary.to_csv(summary_path, index=False)
fold_metrics.to_csv(fold_path, index=False)
predictions.to_csv(pred_path, index=False)
well_diagnostics.to_csv(well_path, index=False)
beta_summary.to_csv(beta_path, index=False)
findings.to_csv(findings_path, index=False)

report = f"""# Multi-Analog Zone Blend Screen

Run ID: `{RUN_ID}`

Result

Best candidate: `{best_candidate}`

- Same-sample baseline RMSE: **{baseline_rmse:.4f}**
- Best RMSE: **{best_row['rmse_mean']:.4f}**
- Delta vs same-sample baseline: **{best_row['rmse_delta_vs_sample_baseline']:.4f}**
- Current full local reference: **{LOCAL_REFERENCE_RMSE:.4f}**
- Current public reference: **{PUBLIC_REFERENCE_RMSE:.3f}**

Interpretation

This screen tests whether nearest-analog transfer improves when multiple analog wells or pseudo-zone-specific blending are used. It is still a residual-sample diagnostic, not a Kaggle submission notebook.
"""
report_path.write_text(report, encoding="utf-8")

comparison_row = pd.DataFrame(
    [
        {
            "run_id": RUN_ID,
            "model_family": "alignment_multi_analog",
            "model_name": best_candidate,
            "cv_rmse_mean": float(best_row["rmse_mean"]),
            "cv_rmse_std": float(best_row["rmse_std"]),
            "train_time_s": float(train_time_s),
            "n_features": int(len(ANALOG_DESCRIPTOR_FEATURES)),
            "notes": "Fold-safe multi-analog and pseudo-zone blend screen on Huber + additive Hilbert residual sample",
        }
    ]
)
comparison = pd.read_csv(comparison_path) if comparison_path.exists() else pd.DataFrame()
if len(comparison):
    comparison = comparison.loc[~comparison["run_id"].eq(RUN_ID)]
comparison = pd.concat([comparison, comparison_row], ignore_index=True)
comparison.to_csv(comparison_path, index=False)

for src in [summary_path, fold_path, pred_path, well_path, beta_path, findings_path, report_path, comparison_path]:
    shutil.copy2(src, ARTIFACT_LOGS / src.name)

config = {"run_id": RUN_ID, "target_candidate": TARGET_CANDIDATE, "offset_clip": OFFSET_CLIP, "candidates": CANDIDATES, "analog_descriptor_features": ANALOG_DESCRIPTOR_FEATURES, "local_reference_rmse": LOCAL_REFERENCE_RMSE, "public_reference_rmse": PUBLIC_REFERENCE_RMSE}
(ARTIFACT_LOGS / "config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")

print(findings.to_string(index=False))
print(f"wrote {summary_path}")
print(f"wrote run bundle {ARTIFACT_LOGS}")

              finding                                                     value                                                       modeling_implication
       best_candidate weighted_k12_s150 RMSE 14.7034 vs sample baseline 14.7439 Promote only if the 0.05 RMSE gate clears and fold diagnostics are stable.
sample_baseline_delta                              best candidate delta -0.0405 A stronger direct analog gain supports a Kaggle-safe analog feature build.
     well_bias_change                   median absolute well-bias delta -0.0870      Negative is good; positive indicates analog blending is adding noise.
wrote C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\artifacts\tables\model__multi_analog_zone_blend_summary__2026-06-23.csv
wrote run bundle C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\artifacts\logs\runs\2026-06-23__multi_analog_zone_blend


## 6. Conclusion

In [6]:
"Print final decision summary."

print(f"best_candidate={best_candidate}")
print(f"sample_baseline_rmse={baseline_rmse:.4f}")
print(f"best_rmse={best_row['rmse_mean']:.4f}")
print(f"delta_vs_sample_baseline={best_row['rmse_delta_vs_sample_baseline']:.4f}")
print(f"promotion_gate_cleared={best_row['rmse_delta_vs_sample_baseline'] <= -0.05}")

best_candidate=weighted_k12_s150
sample_baseline_rmse=14.7439
best_rmse=14.7034
delta_vs_sample_baseline=-0.0405
promotion_gate_cleared=False
